# Prediction of a new Patient

This notebook trains the model and classifies example patients as diseased or healthy.

This is a learning project, not a medical tool!

In [1]:
import sys
from pathlib import Path

import torch
import pandas as pd

# Make the modules in src/ importable from the notebooks/ folder
sys.path.append(str(Path.cwd().parent / "src"))

from config import DROPOUT, INPUT_SIZE
from model import HDD
from data import load_data, preprocessing

/Users/mr/Code/AI/heart-disease-diagnosis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Data
Loading and preprocessing the data.

In [2]:
df = load_data()
(train_loader, val_loader, test_loader), scaler = preprocessing(df)

# 2. Training
`fit` uses early stopping and restores the weight of the best epoch.

In [3]:
model = HDD(input_size=INPUT_SIZE, dropout=DROPOUT)
model = model.fit(train_loader, val_loader)

Training, using the Device: mps
Epoch   0 | train 0.6159 | val 0.5750
Epoch   1 | train 0.5745 | val 0.5579
Epoch   2 | train 0.5648 | val 0.5546
Epoch   3 | train 0.5609 | val 0.5515
Epoch   4 | train 0.5583 | val 0.5502
Epoch   5 | train 0.5562 | val 0.5505
Epoch   6 | train 0.5562 | val 0.5490
Epoch   7 | train 0.5549 | val 0.5489
Epoch   8 | train 0.5534 | val 0.5497
Epoch   9 | train 0.5535 | val 0.5476
Epoch  10 | train 0.5523 | val 0.5478
Epoch  11 | train 0.5517 | val 0.5464
Epoch  12 | train 0.5505 | val 0.5471
Epoch  13 | train 0.5511 | val 0.5466
Epoch  14 | train 0.5504 | val 0.5463
Epoch  15 | train 0.5494 | val 0.5488
Epoch  16 | train 0.5496 | val 0.5461
Epoch  17 | train 0.5505 | val 0.5462
Epoch  18 | train 0.5492 | val 0.5462
Epoch  19 | train 0.5499 | val 0.5459
Epoch  20 | train 0.5503 | val 0.5454
Epoch  21 | train 0.5492 | val 0.5450
Epoch  22 | train 0.5496 | val 0.5450
Epoch  23 | train 0.5492 | val 0.5439
Epoch  24 | train 0.5480 | val 0.5441
Epoch  25 | train 

# 3. Example Patients
A patient is a dictionary with one value per feature.

Feature Meaning (check Kaggle dataset for more details):
-  `age`: in **days** (therefore years * 365)
- `gender`: 1 = female, 2 = male
- `height` in cm, `weight` in kg
- `ap_hi`: systolic blood pressure, `ap_lo`: diastolic blood pressure
- `cholesterol`, `gluc`: 1 = normal, 2 = above normal, 3 = well above normal
- `smoke`, `alco`, `active`: 0 = no, 1 = yes

In [ ]:
patients = {
    "Patient A (35 years, no risk factors)": {
        "age": 35 * 365, "gender": 1, "height": 170, "weight": 65,
        "ap_hi": 115, "ap_lo": 75, "cholesterol": 1, "gluc": 1,
        "smoke": 0, "alco": 0, "active": 1,
    },
    "Patient B (62 years, several risk factors)": {
        "age": 62 * 365, "gender": 2, "height": 175, "weight": 95,
        "ap_hi": 160, "ap_lo": 100, "cholesterol": 3, "gluc": 2,
        "smoke": 1, "alco": 0, "active": 0,
    },
}

# 4. Prediction of Patients
Predict the probability of a heart disease for the patients and classify accordingly.

In [5]:
for name, patient in patients.items():
    prob, diseased = model.predict(patient,scaler)
    print(f"{name}: P(disease) = {prob:.1%} -> {'diseased' if diseased else 'healthy'}")

Patient A (35 years, no risk factors): P(disease) = 4.6% -> healthy
Patient B (62 years, several risk factors): P(disease) = 84.8% -> diseased
